In [1]:
!pip install tensorflow-model-optimization
#warning! This runs ont eh 3.8 -- AzureML kernel by default... 
!conda install tf-keras
#need tf-keras for tf-model-optimization to work

Jupyter detected...

CondaToSNonInteractiveError: Terms of Service have not been accepted for the following channels. Please accept or remove them before proceeding:
    • https://repo.anaconda.com/pkgs/main
    • https://repo.anaconda.com/pkgs/r

To accept a channel's Terms of Service, run the following and replace `CHANNEL` with the channel name/URL:
    ‣ conda tos accept --override-channels --channel CHANNEL

To remove channels with rejected Terms of Service, run the following and replace `CHANNEL` with the channel name/URL:
    ‣ conda config --remove channels CHANNEL



In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist

# Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# Build a simple model
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

2026-03-05 20:54:56.242578: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-05 20:54:56.301898: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-05 20:54:58.094032: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/keras/src/layers/reshap

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8635 - loss: 0.4763 - val_accuracy: 0.9585 - val_loss: 0.1418
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9559 - loss: 0.1514 - val_accuracy: 0.9676 - val_loss: 0.1108
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9671 - loss: 0.1106 - val_accuracy: 0.9722 - val_loss: 0.0883
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9723 - loss: 0.0885 - val_accuracy: 0.9768 - val_loss: 0.0829
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9790 - loss: 0.0703 - val_accuracy: 0.9765 - val_loss: 0.0775


In [3]:
import tensorflow as tf
import tensorflow_model_optimization as tfmot

# 1. FORCE the use of the Keras version tfmot is using
# This bypasses tf.keras and standalone keras
from tensorflow_model_optimization.python.core.keras.compat import keras

# 2. Build the model using THAT specific keras instance
inputs = keras.Input(shape=(28, 28))
x = keras.layers.Flatten()(inputs)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(10, activation='softmax')(x)

base_model = keras.Model(inputs=inputs, outputs=outputs)

# 3. Define pruning parameters
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0, 
        final_sparsity=0.5, 
        begin_step=0, 
        end_step=1000)
}

# 4. Apply pruning
# Since base_model was built with tfmot's 'compat' keras, this WILL pass the type check
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(base_model, **pruning_params)

# 5. Compile and Train
pruned_model.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]
pruned_model.fit(x_train, y_train, epochs=2, validation_data=(x_test, y_test), callbacks=callbacks)

Epoch 1/2
1875/1875 [==============================] - 6s 2ms/step - loss: 0.3165 - accuracy: 0.9093 - val_loss: 0.1699 - val_accuracy: 0.9478
Epoch 2/2
1875/1875 [==============================] - 4s 2ms/step - loss: 0.1729 - accuracy: 0.9492 - val_loss: 0.1223 - val_accuracy: 0.9627


In [4]:
# Convert the pruned model to a TensorFlow Lite quantized model
converter = tf.lite.TFLiteConverter.from_keras_model(pruned_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_model = converter.convert()

INFO:tensorflow:Assets written to: /tmp/tmpmagtrg5i/assets


INFO:tensorflow:Assets written to: /tmp/tmpmagtrg5i/assets
W0000 00:00:1772744255.626809   12453 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1772744255.626842   12453 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-03-05 20:57:35.627130: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpmagtrg5i
2026-03-05 20:57:35.630183: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-05 20:57:35.630195: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpmagtrg5i
I0000 00:00:1772744255.643400   12453 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-03-05 20:57:35.645717: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-05 20:57:35.693482: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpmagtrg5i
2026-03-05 20:57:35.708

In [5]:
# Measure accuracy of the quantized model using the test set
interpreter = tf.lite.Interpreter(model_content=quantized_model)
interpreter.allocate_tensors()

input_index = interpreter.get_input_details()[0]['index']
output_index = interpreter.get_output_details()[0]['index']

# Evaluate accuracy
correct_predictions = 0
for i in range(len(x_test)):
    input_data = x_test[i:i+1].astype('float32')
    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_index)
    predicted_label = output.argmax()
    if predicted_label == y_test[i]:
        correct_predictions += 1

accuracy = correct_predictions / len(x_test)
print(f'Quantized model accuracy: {accuracy * 100:.2f}%')

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Quantized model accuracy: 96.27%
